#### ============================================================
#### 02 PREPROCESSING NOTEBOOK
#### Dissertation Project:
#### Multilingual Emotion Mining and Topic Modelling of Public Unrest on X
#### Case Studies: #FeesMustFall and #EndSARS
#### ============================================================


In [1]:
import pandas as pd
import numpy as np
import os
import re
import html
from ftfy import fix_text

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
# 1. DEFINE PATHS


PROJECT_DIR = r"C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT"

RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")

TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
PROCESSED_DIR = os.path.join(OUTPUT_DIR, "processed")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

FEES_PATH = os.path.join(RAW_DIR, "FeesMustFall (V MODEL CODE BOOK).xlsx")
ENDSARS_PATH = os.path.join(RAW_DIR, "endsars.xlsx")

print(os.path.exists(FEES_PATH), FEES_PATH)
print(os.path.exists(ENDSARS_PATH), ENDSARS_PATH)

True C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\data\raw\FeesMustFall (V MODEL CODE BOOK).xlsx
True C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\data\raw\endsars.xlsx


In [3]:
# 2. LOAD DATASETS

fees = pd.read_excel(
    FEES_PATH,
    sheet_name="1. FeesMustFall"
)

endsars = pd.read_excel(
    ENDSARS_PATH
)

print("FeesMustFall:", fees.shape)
print("EndSARS:", endsars.shape)

FeesMustFall: (462770, 27)
EndSARS: (131930, 9)


In [4]:
# 3. STANDARDISE FEESMUSTFALL

fees_std = fees[[
    "date_time",
    "tweet_id",
    "clear_text",
    "user_screen_name",
    "user_location",
    "user_verified",
    "retweets",
    "favorites",
    "tweet_language",
    "hashtags"
]].copy()

fees_std = fees_std.rename(columns={
    "date_time": "date",
    "clear_text": "original_text",
    "user_screen_name": "username",
    "user_location": "location",
    "user_verified": "verified",
    "favorites": "likes",
    "tweet_language": "source_language"
})

fees_std["movement"] = "FeesMustFall"

fees_std.head()

,date,tweet_id,original_text,username,location,verified,retweets,likes,source_language,hashtags,movement
0,2015-03-21 14:28:07,5.792884e+17,Priorities?? #FeesMustFall RT @informer_sa: UCT's #RhodesMustFall campaign gains momentum http://t.co/1eIIX3mWZx http://t.co/5FY78MKzjI,SkhumbuzoTuswa,"Braamfontein, South Africa",0.0,4,5.0,en,#feesmustfall #rhodesmustfall,FeesMustFall
1,2015-04-07 04:31:56,5.852989e+17,#FEESMustFall that will make sense to me. Free quality education in our lifetime,SSSIBIYA,Nkandla eMagidini,0.0,2,4.0,en,#feesmustfall,FeesMustFall
2,2015-10-13 17:07:34,6.539804e+17,Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus #FeesMustFall http://t.co/9aO790bbIr,MadVocate_,South Africa,0.0,6,1.0,en,#feesmustfall,FeesMustFall
3,2015-10-13 17:13:32,6.539819e+17,Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. #FeesMustFall,MadVocate_,South Africa,0.0,5,3.0,en,#feesmustfall,FeesMustFall
4,2015-10-13 17:14:52,6.539822e+17,Tomorrow is D-Day. Sikhathele! #FeesMustFall http://t.co/JalbjHoPJL,MadVocate_,South Africa,0.0,4,2.0,en,#feesmustfall,FeesMustFall


In [5]:
# 4. STANDARDISE ENDSARS

endsars_std = endsars[[
    "Datetime",
    "Tweet Id",
    "Text",
    "Username",
    "location",
    "verified",
    "retweet",
    "likes"
]].copy()

endsars_std = endsars_std.rename(columns={
    "Datetime": "date",
    "Tweet Id": "tweet_id",
    "Text": "original_text",
    "Username": "username",
    "retweet": "retweets"
})

endsars_std["movement"] = "EndSARS"
endsars_std["source_language"] = np.nan
endsars_std["hashtags"] = np.nan

endsars_std.head()

,date,tweet_id,original_text,username,location,verified,retweets,likes,movement,source_language,hashtags
0,2020-12-30 23:59:39+00:00,1344432997957850112,"@DannyWalta In 2020, politicians in Nig almost unwittingly killed about 40 million Nigerians through starvation, by locking up warehouses stuffed with staples. #EndSARSÂ Â indirectly opened these...",LeagueofCitize1,Nigeria,False,1,0,EndSARS,NaN,NaN
1,2020-12-30 23:59:36+00:00,1344432982896110080,31-12-2020 Rounding Upâ€¦ \n\n1-1-2021 Loadingâ€¦ \n\nDear Passengers Exercise Some Patience. ðŸ™‚\n\n#TuleChallenge #Nadeco #Thiago #Klopp #BuhariTrain #Psquare #Wizkidfc #NoStoryWithGuinness #En...,Mc_Hugo92,"Lagos, Nigeria",False,1,1,EndSARS,NaN,NaN
2,2020-12-30 23:59:11+00:00,1344432881486139904,#EndSARS,JarrodB19_,"Michigan, USA",False,2,2,EndSARS,NaN,NaN
3,2020-12-30 23:59:11+00:00,1344432877975559936,"@viralpostsng In 2020, politicians in Nig almost unwittingly killed about 40 million Nigerians through starvation, by locking up warehouses stuffed with staples. #EndSARSÂ Â indirectly opened the...",LeagueofCitize1,Nigeria,False,1,0,EndSARS,NaN,NaN
4,2020-12-30 23:58:46+00:00,1344432775747820032,"@Harryolah In 2020, politicians in Nig almost unwittingly killed about 40 million Nigerians through starvation, by locking up warehouses stuffed with staples. #EndSARSÂ Â indirectly opened these ...",LeagueofCitize1,Nigeria,False,1,1,EndSARS,NaN,NaN


In [6]:
# 5. COMBINE DATASETS

combined = pd.concat(
    [fees_std, endsars_std],
    ignore_index=True
)

print("Combined shape:", combined.shape)
print(combined["movement"].value_counts())

Combined shape: (594700, 11)
movement
FeesMustFall    462770
EndSARS         131930
Name: count, dtype: int64


In [7]:
# 6. FIX DATA TYPES

combined["date"] = pd.to_datetime(
    combined["date"],
    errors="coerce",
    utc=True
)

combined["retweets"] = pd.to_numeric(
    combined["retweets"],
    errors="coerce"
).fillna(0).astype(int)

combined["likes"] = pd.to_numeric(
    combined["likes"],
    errors="coerce"
).fillna(0).astype(int)

combined["verified"] = combined["verified"].fillna(False).astype(bool)

combined.dtypes

date               datetime64[ns, UTC]
tweet_id                       float64
original_text                   object
username                        object
location                        object
verified                          bool
retweets                         int64
likes                            int64
source_language                 object
hashtags                        object
movement                        object
dtype: object

In [8]:
# 7. REMOVE MISSING TEXT

combined["original_text"] = combined["original_text"].astype(str)

combined["original_text"] = combined["original_text"].replace(
    ["nan", "None", ""],
    np.nan
)

missing_text_before = combined["original_text"].isna().sum()

combined = combined.dropna(subset=["original_text"]).copy()

print("Missing text removed:", missing_text_before)
print("Remaining rows:", combined.shape)

Missing text removed: 7
Remaining rows: (594693, 11)


In [9]:
# 8. REMOVE EXACT DUPLICATES WITHIN EACH MOVEMENT


before_duplicates = combined.shape[0]

combined = combined.drop_duplicates(
    subset=["movement", "original_text"],
    keep="first"
).copy()

after_duplicates = combined.shape[0]
removed_duplicates = before_duplicates - after_duplicates

print("Duplicates removed:", removed_duplicates)
print("Remaining rows:", combined.shape)

Duplicates removed: 19080
Remaining rows: (575613, 11)


In [10]:
# 9. TEXT REPAIR AND CLEANING FUNCTIONS

def repair_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = fix_text(text)
    text = html.unescape(text)
    return text


def extract_hashtags(text):
    if pd.isna(text):
        return []
    return re.findall(r"#\w+", str(text))


def clean_tweet_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Remove user mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove RT marker
    text = re.sub(r"\bRT\b", " ", text)

    # Remove hashtag symbol but keep hashtag words
    text = re.sub(r"#", "", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
# 10. APPLY REPAIR, HASHTAG EXTRACTION, AND CLEANING

combined["repaired_text"] = combined["original_text"].apply(repair_text)

combined["extracted_hashtags"] = combined["repaired_text"].apply(extract_hashtags)

combined["clean_text"] = combined["repaired_text"].apply(clean_tweet_text)

combined["clean_text_length"] = combined["clean_text"].str.len()

combined["word_count"] = combined["clean_text"].str.split().str.len()

combined[[
    "movement",
    "original_text",
    "repaired_text",
    "extracted_hashtags",
    "clean_text"
]].head(10)

,movement,original_text,repaired_text,extracted_hashtags,clean_text
0,FeesMustFall,Priorities?? #FeesMustFall RT @informer_sa: UCT's #RhodesMustFall campaign gains momentum http://t.co/1eIIX3mWZx http://t.co/5FY78MKzjI,Priorities?? #FeesMustFall RT @informer_sa: UCT's #RhodesMustFall campaign gains momentum http://t.co/1eIIX3mWZx http://t.co/5FY78MKzjI,"[#FeesMustFall, #RhodesMustFall]",Priorities?? FeesMustFall : UCT's RhodesMustFall campaign gains momentum
1,FeesMustFall,#FEESMustFall that will make sense to me. Free quality education in our lifetime,#FEESMustFall that will make sense to me. Free quality education in our lifetime,[#FEESMustFall],FEESMustFall that will make sense to me. Free quality education in our lifetime
2,FeesMustFall,Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus #FeesMustFall http://t.co/9aO790bbIr,Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus #FeesMustFall http://t.co/9aO790bbIr,[#FeesMustFall],Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus FeesMustFall
3,FeesMustFall,Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. #FeesMustFall,Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. #FeesMustFall,[#FeesMustFall],Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. FeesMustFall
4,FeesMustFall,Tomorrow is D-Day. Sikhathele! #FeesMustFall http://t.co/JalbjHoPJL,Tomorrow is D-Day. Sikhathele! #FeesMustFall http://t.co/JalbjHoPJL,[#FeesMustFall],Tomorrow is D-Day. Sikhathele! FeesMustFall
5,FeesMustFall,NO WE ARE NOT!!! #FEESMustFall https://t.co/TU19G2v3Yd,NO WE ARE NOT!!! #FEESMustFall https://t.co/TU19G2v3Yd,[#FEESMustFall],NO WE ARE NOT!!! FEESMustFall
6,FeesMustFall,If we don't take a stand...Our futures won't be able to stand! #FeesMustFall #PYA #WITS,If we don't take a stand...Our futures won't be able to stand! #FeesMustFall #PYA #WITS,"[#FeesMustFall, #PYA, #WITS]",If we don't take a stand...Our futures won't be able to stand! FeesMustFall PYA WITS
7,FeesMustFall,That's why EFF is saying nfsas must fall universities must be free #feesmustfall,That's why EFF is saying nfsas must fall universities must be free #feesmustfall,[#feesmustfall],That's why EFF is saying nfsas must fall universities must be free feesmustfall
8,FeesMustFall,"Scrap #WitsFeesMustFall #FeesMustFall nje we want Free Education, believe me when I say it is possible in SA aai umontomnyama bakhithi Noo,","Scrap #WitsFeesMustFall #FeesMustFall nje we want Free Education, believe me when I say it is possible in SA aai umontomnyama bakhithi Noo,","[#WitsFeesMustFall, #FeesMustFall]","Scrap WitsFeesMustFall FeesMustFall nje we want Free Education, believe me when I say it is possible in SA aai umontomnyama bakhithi Noo,"
9,FeesMustFall,Current situation at Wits University #AlutaContinua #FEESMUSTFALL #feesincreasestrike http://t.co/mtcAFHEEHz,Current situation at Wits University #AlutaContinua #FEESMUSTFALL #feesincreasestrike http://t.co/mtcAFHEEHz,"[#AlutaContinua, #FEESMUSTFALL, #feesincreasestrike]",Current situation at Wits University AlutaContinua FEESMUSTFALL feesincreasestrike


In [12]:
# 11. REMOVE EMPTY OR VERY SHORT CLEANED TEXT

before_cleaning_filter = combined.shape[0]

combined = combined[
    (combined["clean_text_length"] > 2) &
    (combined["word_count"] >= 3)
].copy()

removed_after_cleaning = before_cleaning_filter - combined.shape[0]

print("Rows removed after cleaning filters:", removed_after_cleaning)
print("Remaining rows:", combined.shape)

Rows removed after cleaning filters: 37515
Remaining rows: (538098, 16)


In [13]:
# 12. REMOVE LOW-INFORMATION TWEETS
# Example: "EndSARS EndSARS EndSARS"

combined["unique_word_count"] = (
    combined["clean_text"]
    .str.lower()
    .str.split()
    .apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
)

before_low_info = combined.shape[0]

combined = combined[
    (combined["unique_word_count"] >= 3) &
    (combined["word_count"] >= 4)
].copy()

removed_low_info = before_low_info - combined.shape[0]

print("Low-information tweets removed:", removed_low_info)
print("Remaining rows:", combined.shape)

Low-information tweets removed: 17227
Remaining rows: (520871, 17)


In [14]:
# 13. CREATE EXTRA FEATURES

combined["year"] = combined["date"].dt.year
combined["month"] = combined["date"].dt.to_period("M").astype(str)
combined["day"] = combined["date"].dt.date

combined["has_hashtag"] = combined["extracted_hashtags"].apply(lambda x: len(x) > 0)
combined["has_location"] = combined["location"].notna()

combined.head()

C:\Users\emeka\AppData\Local\Temp\ipykernel_23876\1123289294.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  combined["month"] = combined["date"].dt.to_period("M").astype(str)


,date,tweet_id,original_text,username,location,verified,retweets,likes,source_language,hashtags,movement,repaired_text,extracted_hashtags,clean_text,clean_text_length,word_count,unique_word_count,year,month,day,has_hashtag,has_location
0,2015-03-21 14:28:07+00:00,5.792884e+17,Priorities?? #FeesMustFall RT @informer_sa: UCT's #RhodesMustFall campaign gains momentum http://t.co/1eIIX3mWZx http://t.co/5FY78MKzjI,SkhumbuzoTuswa,"Braamfontein, South Africa",False,4,5,en,#feesmustfall #rhodesmustfall,FeesMustFall,Priorities?? #FeesMustFall RT @informer_sa: UCT's #RhodesMustFall campaign gains momentum http://t.co/1eIIX3mWZx http://t.co/5FY78MKzjI,"[#FeesMustFall, #RhodesMustFall]",Priorities?? FeesMustFall : UCT's RhodesMustFall campaign gains momentum,72,8,8,2015,2015-03,2015-03-21,True,True
1,2015-04-07 04:31:56+00:00,5.852989e+17,#FEESMustFall that will make sense to me. Free quality education in our lifetime,SSSIBIYA,Nkandla eMagidini,False,2,4,en,#feesmustfall,FeesMustFall,#FEESMustFall that will make sense to me. Free quality education in our lifetime,[#FEESMustFall],FEESMustFall that will make sense to me. Free quality education in our lifetime,79,13,13,2015,2015-04,2015-04-07,True,True
2,2015-10-13 17:07:34+00:00,6.539804e+17,Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus #FeesMustFall http://t.co/9aO790bbIr,MadVocate_,South Africa,False,6,1,en,#feesmustfall,FeesMustFall,Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus #FeesMustFall http://t.co/9aO790bbIr,[#FeesMustFall],Now it is time for me to mobilize Wits students. Tomorrow at 12pm we meet at West campus FeesMustFall,101,19,18,2015,2015-10,2015-10-13,True,True
3,2015-10-13 17:13:32+00:00,6.539819e+17,Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. #FeesMustFall,MadVocate_,South Africa,False,5,3,en,#feesmustfall,FeesMustFall,Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. #FeesMustFall,[#FeesMustFall],Residence is going up by over 9% yet I always walk up 18 flights of stairs because lifts don't work at res. FeesMustFall,120,23,22,2015,2015-10,2015-10-13,True,True
4,2015-10-13 17:14:52+00:00,6.539822e+17,Tomorrow is D-Day. Sikhathele! #FeesMustFall http://t.co/JalbjHoPJL,MadVocate_,South Africa,False,4,2,en,#feesmustfall,FeesMustFall,Tomorrow is D-Day. Sikhathele! #FeesMustFall http://t.co/JalbjHoPJL,[#FeesMustFall],Tomorrow is D-Day. Sikhathele! FeesMustFall,43,5,5,2015,2015-10,2015-10-13,True,True


In [15]:
# 14. PREPROCESSING SUMMARY

preprocessing_summary = pd.DataFrame({
    "Metric": [
        "Initial combined rows",
        "Missing text removed",
        "Duplicate rows removed",
        "Rows removed after cleaning filters",
        "Low-information tweets removed",
        "Final processed rows"
    ],
    "Value": [
        fees.shape[0] + endsars.shape[0],
        missing_text_before,
        removed_duplicates,
        removed_after_cleaning,
        removed_low_info,
        combined.shape[0]
    ]
})

preprocessing_summary

,Metric,Value
0,Initial combined rows,594700
1,Missing text removed,7
2,Duplicate rows removed,19080
3,Rows removed after cleaning filters,37515
4,Low-information tweets removed,17227
5,Final processed rows,520871


In [16]:
preprocessing_summary.to_csv(
    os.path.join(TABLE_DIR, "table_11_preprocessing_summary.csv"),
    index=False
)

In [17]:
# 15. SAVE FULL CLEANED CORPUS

full_processed_path = os.path.join(
    PROCESSED_DIR,
    "processed_combined_corpus_cleaned.csv"
)

combined.to_csv(
    full_processed_path,
    index=False,
    encoding="utf-8"
)

print("Full cleaned corpus saved to:", full_processed_path)
print("Full cleaned corpus shape:", combined.shape)

Full cleaned corpus saved to: C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\outputs\processed\processed_combined_corpus_cleaned.csv
Full cleaned corpus shape: (520871, 22)


In [18]:
# 16. SAMPLE HIERARCHY
# Main modelling corpus: 200k
# Development corpus: 100k
# Quick experiment corpus: 40k


MAIN_SAMPLE_SIZE = 100000
DEV_SAMPLE_SIZE = 50000
QUICK_SAMPLE_SIZE = 20000

main_corpus = combined.groupby(
    "movement",
    group_keys=False
).sample(
    n=MAIN_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

development_corpus = main_corpus.groupby(
    "movement",
    group_keys=False
).sample(
    n=DEV_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

quick_experiment_corpus = main_corpus.groupby(
    "movement",
    group_keys=False
).sample(
    n=QUICK_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

print("Main corpus:")
print(main_corpus.shape)
print(main_corpus["movement"].value_counts())

print("\nDevelopment corpus:")
print(development_corpus.shape)
print(development_corpus["movement"].value_counts())

print("\nQuick experiment corpus:")
print(quick_experiment_corpus.shape)
print(quick_experiment_corpus["movement"].value_counts())

Main corpus:
(200000, 22)
movement
EndSARS         100000
FeesMustFall    100000
Name: count, dtype: int64

Development corpus:
(100000, 22)
movement
EndSARS         50000
FeesMustFall    50000
Name: count, dtype: int64

Quick experiment corpus:
(40000, 22)
movement
EndSARS         20000
FeesMustFall    20000
Name: count, dtype: int64


In [19]:
# 17. SAVE SAMPLE HIERARCHY

main_corpus_path = os.path.join(
    PROCESSED_DIR,
    "main_modelling_corpus_200k.csv"
)

development_corpus_path = os.path.join(
    PROCESSED_DIR,
    "development_corpus_100k.csv"
)

quick_experiment_path = os.path.join(
    PROCESSED_DIR,
    "quick_experiment_corpus_40k.csv"
)

main_corpus.to_csv(
    main_corpus_path,
    index=False,
    encoding="utf-8"
)

development_corpus.to_csv(
    development_corpus_path,
    index=False,
    encoding="utf-8"
)

quick_experiment_corpus.to_csv(
    quick_experiment_path,
    index=False,
    encoding="utf-8"
)

print("Saved files:")
print(main_corpus_path)
print(development_corpus_path)
print(quick_experiment_path)

Saved files:
C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\outputs\processed\main_modelling_corpus_200k.csv
C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\outputs\processed\development_corpus_100k.csv
C:\Users\emeka\OneDrive - hull.ac.uk\DISSERTATION PROJECT\outputs\processed\quick_experiment_corpus_40k.csv


In [20]:
# 18. FINAL CHECK

print("PREPROCESSING COMPLETE")
print("-" * 50)

print("Full cleaned corpus:", combined.shape)
print("Main modelling corpus:", main_corpus.shape)
print("Development corpus:", development_corpus.shape)
print("Quick experiment corpus:", quick_experiment_corpus.shape)

print("\nMovement distribution in main corpus:")
print(main_corpus["movement"].value_counts())

print("\nMovement distribution in development corpus:")
print(development_corpus["movement"].value_counts())

print("\nMovement distribution in quick experiment corpus:")
print(quick_experiment_corpus["movement"].value_counts())

PREPROCESSING COMPLETE
--------------------------------------------------
Full cleaned corpus: (520871, 22)
Main modelling corpus: (200000, 22)
Development corpus: (100000, 22)
Quick experiment corpus: (40000, 22)

Movement distribution in main corpus:
movement
EndSARS         100000
FeesMustFall    100000
Name: count, dtype: int64

Movement distribution in development corpus:
movement
EndSARS         50000
FeesMustFall    50000
Name: count, dtype: int64

Movement distribution in quick experiment corpus:
movement
EndSARS         20000
FeesMustFall    20000
Name: count, dtype: int64


In [1]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\emeka\AppData\Local\Programs\Python\Python313\python.exe
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
